# Basit Adapter Testi

Bu notebook Notebook 3'e dokunmadan daha basit bir kullanici akisi sunar.

Kullanim:
1. Ilk kod hucresini bir kez calistirin.
2. Ikinci kod hucresini bir kez calistirin.
3. Acilan arayuzden adapter secin, resmi yukleyin ve `Tahmin Et` butonuna basin.
4. Yeni tahmin icin hucreyi tekrar calistirmayin; panelde `Yeni Resim` veya `Resim Yukle` ile gorseli degistirip tekrar `Tahmin Et` basin.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    collect_notebook_access_report,
    install_colab_requirements,
    print_notebook_access_report,
    resolve_repo_root,
    running_in_colab,
)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if running_in_colab():
    install_colab_requirements(ROOT / 'requirements_colab.txt', in_colab=True)

from src.core.config_manager import get_config

CONFIG_FOR_ACCESS = get_config(environment='colab')
BACKBONE_MODEL_NAME = str(dict(dict(CONFIG_FOR_ACCESS.get('training', {})).get('continual', {})).get('backbone', {}).get('model_name', '')).strip()
ACCESS_REPORT = collect_notebook_access_report(
    repo_root=ROOT,
    hf_model_ids=[BACKBONE_MODEL_NAME] if BACKBONE_MODEL_NAME else [],
)
print_notebook_access_report(ACCESS_REPORT, print_fn=print)
if BACKBONE_MODEL_NAME:
    print(f"[KONTROL] Varsayilan backbone modeli: {BACKBONE_MODEL_NAME}")
print(ROOT)


In [ ]:
import importlib
from scripts import colab_simple_adapter_smoke_ui

colab_simple_adapter_smoke_ui = importlib.reload(colab_simple_adapter_smoke_ui)
launch_simple_adapter_smoke_ui = colab_simple_adapter_smoke_ui.launch_simple_adapter_smoke_ui

launch_simple_adapter_smoke_ui(ROOT)
